# Assignment 02 — Singular Value Decomposition (100 points)

**Unit**: AI 200 — Mathematical Foundations for AI  
**Competition alignment**: USAAIO 2026 Round 1 & Round 2  

---

## Background

The SVD factorizes any $m \times n$ matrix as $A = U\Sigma V^\top$ where $U \in \mathbb{R}^{m \times m}$ and $V \in \mathbb{R}^{n \times n}$ are orthogonal and $\Sigma$ is diagonal with non-negative entries (singular values) in decreasing order. The compact SVD keeps only the $r = \text{rank}(A)$ nonzero singular values. SVD is the workhorse behind PCA, pseudoinverses, low-rank approximation, and recommender systems.

## Notation

| Symbol | Meaning |
|--------|---------|
| $A$ | data matrix, shape $(m, n)$ |
| $U$ | left singular vectors, shape $(m, r)$ (compact) |
| $\Sigma$ | diagonal singular values, shape $(r, r)$ |
| $V^\top$ | right singular vectors transposed, shape $(r, n)$ |
| $\sigma_i$ | $i$-th singular value, $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ |
| $A_k$ | best rank-$k$ approximation of $A$ |

In [ ]:
# DO NOT MAKE ANY CHANGE IN THIS CELL
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)

> **WARNING**: Do not import any additional libraries. For Parts 1-2 you may only use `np.linalg.eigh` (not `np.linalg.svd`). For Parts 3-4 you may use `np.linalg.svd` for efficiency.

---

## Part 1 (35 points, coding task)

**SVD from Scratch**

Compute the compact SVD $A = U\Sigma V^\top$ without using `np.linalg.svd`.

**Algorithm**:
1. Form $A^\top A$ — shape $(n, n)$
2. Eigendecompose: $A^\top A = V \text{diag}(\sigma_i^2) V^\top$ using `np.linalg.eigh`
3. Singular values: $\sigma_i = \sqrt{\max(\lambda_i, 0)}$ — keep only positive ones
4. Sort in descending order
5. Compute $U$ from $\mathbf{u}_i = A\mathbf{v}_i / \sigma_i$

*Reasoning is not required.*

In [ ]:
def svd_from_scratch(A: np.ndarray) -> tuple:
    """Compute the compact SVD of A from scratch.
    
    Args:
        A: shape (m, n)
    
    Returns:
        U: shape (m, r) - left singular vectors
        S: shape (r,) - singular values in descending order
        Vt: shape (r, n) - right singular vectors transposed
        where r = rank(A)
    """
    m, n = A.shape
    
    ### WRITE YOUR SOLUTION HERE ###
    
    pass

""" END OF THIS PART """

In [ ]:
# Test: compare with np.linalg.svd
A = np.random.randn(5, 3)  # (5, 3)

U_scratch, S_scratch, Vt_scratch = svd_from_scratch(A)
U_np, S_np, Vt_np = np.linalg.svd(A, full_matrices=False)

print(f"Singular values (scratch): {S_scratch}")
print(f"Singular values (numpy):   {S_np}")
print(f"Singular value error: {np.linalg.norm(S_scratch - S_np):.2e}")

# Verify reconstruction
A_reconstructed = U_scratch * S_scratch @ Vt_scratch  # (5, 3)
print(f"Reconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}")

---

The Eckart-Young theorem states that the best rank-$k$ approximation (in Frobenius norm) is obtained by keeping only the top $k$ singular values: $A_k = U_k \Sigma_k V_k^\top$. The approximation error equals $\|A - A_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$.

---

## Part 2 (25 points, coding task)

**Truncated SVD and the Eckart-Young Theorem**

Implement rank-$k$ approximation via truncated SVD. Empirically verify the Eckart-Young error formula for all ranks $k = 1, \ldots, \min(m,n)-1$.

*Reasoning is not required.*

In [ ]:
def truncated_svd(A: np.ndarray, k: int) -> tuple:
    """Compute rank-k approximation via truncated SVD.
    
    Args:
        A: shape (m, n)
        k: target rank (1 <= k <= min(m, n))
    
    Returns:
        A_k: shape (m, n) - rank-k approximation
        S: shape (min(m,n),) - all singular values
    """
    ### WRITE YOUR SOLUTION HERE ###
    
    pass

""" END OF THIS PART """

In [ ]:
# Verify Eckart-Young theorem
A = np.random.randn(10, 8)  # (10, 8)

for k in range(1, 8):
    A_k, S = truncated_svd(A, k)
    error_sq = np.linalg.norm(A - A_k, 'fro') ** 2
    expected_sq = np.sum(S[k:] ** 2)
    print(f"Rank-{k}: ||A-A_k||^2 = {error_sq:.6f}, sum(sigma_i^2, i>{k}) = {expected_sq:.6f}, match = {np.isclose(error_sq, expected_sq)}")

---

Low-rank approximation is the basis of compression. A rank-$k$ approximation of an $m \times n$ matrix stores only $k(m + n + 1)$ numbers instead of $mn$, giving a compression ratio of $k(m + n + 1) / (mn)$.

---

## Part 3 (20 points, coding task)

**Image Compression via SVD**

Create a synthetic "image" (a low-rank signal plus noise) and compress it using truncated SVD. For each rank $k \in \{1, 2, 3, 5, 10, 20, 50\}$, compute:
- Compression ratio: $\frac{k(m + n + 1)}{mn}$
- Relative error: $\frac{\|A - A_k\|_F}{\|A\|_F}$

You may use `np.linalg.svd` in this part.

*Reasoning is not required.*

In [ ]:
# Synthetic image: low-rank signal + noise
m, n = 100, 80
true_rank = 5

U_true = np.random.randn(m, true_rank)  # (100, 5)
V_true = np.random.randn(n, true_rank)  # (80, 5)
signal = U_true @ V_true.T              # (100, 80) — rank 5
noise = 0.5 * np.random.randn(m, n)     # (100, 80)
image = signal + noise                   # (100, 80)

### WRITE YOUR SOLUTION HERE ###
# For k in [1, 2, 3, 5, 10, 20, 50]:
#   1. Compute rank-k approximation
#   2. Compute compression ratio: k * (m + n + 1) / (m * n)
#   3. Compute relative error: ||A - A_k||_F / ||A||_F
#   4. Print results

pass

""" END OF THIS PART """

---

The Moore-Penrose pseudoinverse $A^+ = V\Sigma^+ U^\top$ generalizes the matrix inverse to rectangular and rank-deficient matrices. It provides the minimum-norm least-squares solution to $A\mathbf{x} = \mathbf{b}$.

---

## Part 4 (20 points, mixed task)

**Pseudoinverse via SVD**

**(a)** (15 points, coding) Implement the Moore-Penrose pseudoinverse: $A^+ = V\Sigma^+ U^\top$ where $\Sigma^+$ inverts all singular values above a tolerance threshold. Use it to solve an overdetermined least-squares system. *Reasoning is not required.*

**(b)** (5 points, non-coding) Explain why $A^+ \mathbf{b}$ gives the minimum-norm least-squares solution, i.e., why among all minimizers of $\|A\mathbf{x} - \mathbf{b}\|^2$, the pseudoinverse selects the one with smallest $\|\mathbf{x}\|$. *Reasoning is required.*

In [ ]:
# Part 4a
def pseudoinverse(A: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """Compute the Moore-Penrose pseudoinverse via SVD.
    
    Args:
        A: shape (m, n)
        tol: threshold for treating singular values as zero
    
    Returns:
        A_pinv: shape (n, m)
    """
    ### WRITE YOUR SOLUTION HERE ###
    
    pass

In [ ]:
# Test: overdetermined least squares — 10 equations, 3 unknowns
X = np.random.randn(10, 3)                          # (10, 3)
w_true = np.array([1, -2, 3], dtype=float)           # (3,)
y = X @ w_true + 0.1 * np.random.randn(10)          # (10,)

X_pinv = pseudoinverse(X)   # (3, 10)
w_hat = X_pinv @ y          # (3,)

print(f"True weights:      {w_true}")
print(f"Estimated weights: {w_hat}")
print(f"Error: {np.linalg.norm(w_hat - w_true):.6f}")

# Compare with numpy
w_np = np.linalg.pinv(X) @ y  # (3,)
print(f"Numpy pinv result: {w_np}")
print(f"Match: {np.allclose(w_hat, w_np)}")

### WRITE YOUR SOLUTION HERE (Part 4b) ###



""" END OF THIS PART """